## **UNIT 3: Time Series Indexing — DatetimeIndex Deep Dive** 🔴

**Target Time:** 2–2.5 hours of focused work  
**Status:** CRITICAL — core path only, zero detours

---

### **🎯 Why This Matters in Quant Finance**

You're about to learn the most consequential indexing system in finance. Every pricing model, every backtest, every risk calculation depends on **exact temporal alignment**. Get DatetimeIndex wrong and you'll:

- Miss corporate actions by one day (dividends, splits, earnings)
- Calculate returns using mismatched timestamps (US close vs Asia open)
- Resample data incorrectly (weekend gaps, holiday closures)
- Build strategies that work in backtests but fail live (timezone bugs)

**The brutal truth:** Most retail quant failures aren't from bad alpha — they're from off-by-one-day errors that compound silently. This unit prevents that.

---

### **📋 Learning Objectives**

By the end of this unit, you will:

1. **Create DatetimeIndex** from strings, timestamps, and ranges — the three foundational pathways
2. **Extract temporal components** using `.dt` accessor (year, month, day, weekday)
3. **Use partial string indexing** to slice time series (`df['2023-Q3']`, `df['2023-03']`)
4. **Distinguish timezone-naive vs timezone-aware** timestamps and convert between them
5. **Apply business day calendars** (NYSE/LSE) to handle holidays correctly
6. **Validate trading data integrity** (no weekends, no holidays, correct alignment)

---

### **🧠 Core Concepts (Expanded with Context)**

#### **1. DatetimeIndex Creation — Three Pathways**

The foundation of all time series work. You need to master creating DatetimeIndex from:

- **Strings:** `pd.to_datetime(['2023-01-03', '2023-01-04'])` — parsing human-readable dates
- **Timestamps:** `pd.DatetimeIndex([pd.Timestamp('2023-01-03'), ...])` — from existing timestamp objects
- **Ranges:** `pd.date_range('2023-01-01', '2023-12-31', freq='B')` — generating sequences (B = business days)

**Why it matters:** Market data arrives in all three formats. Bloomberg might send timestamps, CSVs have strings, and you generate ranges for backtests.

#### **2. The `.dt` Accessor — Temporal Decomposition**

A Series with datetime values gets the `.dt` accessor to extract components:

```python
df['date'].dt.year      # Extract year
df['date'].dt.month     # Extract month
df['date'].dt.day       # Extract day
df['date'].dt.dayofweek # Monday=0, Sunday=6
df['date'].dt.quarter   # Q1=1, Q2=2, etc.
```

**Critical detail:** `.dt` only works on **Series**, not DatetimeIndex directly. For index: `df.index.year`, for column: `df['date'].dt.year`.

**Why it matters:** You'll filter data by "all Fridays" or "end of quarter" constantly. This is how.

#### **3. Partial String Indexing — The Killer Feature**

DatetimeIndex lets you slice using **partial date strings**:

```python
df['2023']           # All of 2023
df['2023-Q3']        # July-September 2023
df['2023-03']        # All of March 2023
df['2023-03-15']     # Specific day
```

**Why `.loc` works but `.iloc` fails:** `.loc` is label-based (dates are labels), `.iloc` is position-based (integers only).

**Why it matters:** Corporate earnings seasons, quarterly rebalancing, monthly reporting — all use date-based slicing.

#### **4. Timezone-Naive vs Timezone-Aware**

- **Naive:** No timezone info attached. `2023-01-03 09:30:00` — 9:30 *where*?
- **Aware:** Includes timezone. `2023-01-03 09:30:00-05:00` (US/Eastern)

**Conversion:**
```python
df.index = df.index.tz_localize('UTC')        # Naive → Aware (assign timezone)
df.index = df.index.tz_convert('US/Eastern')  # Aware → Aware (change timezone)
```

**The gotcha:** You cannot mix naive and aware timestamps in operations. Pandas will throw `TypeError`.

**Why it matters:** Global portfolios. SPY closes at 16:00 US/Eastern, but Nikkei data is in Asia/Tokyo. Misalignment = wrong returns.

#### **5. Business Day Calendars — Real Trading Days**

Not all days are trading days. Use custom calendars:

```python
from pandas.tseries.offsets import CustomBusinessDay
from pandas.tseries.holiday import USFederalHolidayCalendar

nyse_cal = CustomBusinessDay(calendar=USFederalHolidayCalendar())
trading_days = pd.date_range('2023-01-01', '2023-12-31', freq=nyse_cal)
```

**Why the 252-day assumption is dangerous:** NYSE closes for holidays (Thanksgiving, Christmas, etc.). Assuming 252 days per year introduces **calendar drift** in volatility calculations.

**Why it matters:** Your backtests need to use actual trading days, not calendar days. Otherwise, you're simulating trades on weekends.

#### **6. `.asfreq()` vs `.resample()` — Preview**

Quick distinction (full coverage in Unit 4):

- **`.asfreq('D')`:** Changes frequency to daily, **fills missing dates with NaN** (explicit gaps)
- **`.resample('D').mean()`:** Aggregates data to daily using a function (e.g., mean of all hourly data)

**Why it matters:** Different use cases. `.asfreq()` for alignment, `.resample()` for downsampling.

---

### **💻 Hands-On Tasks (Code Structure Provided)**

Work through these in sequence. Each builds on the previous.

#### **Task 1: Pull Multi-Year Stock Data**

```python
import yfinance as yf

# Pull SPY, AAPL, MSFT from 2019-2024
tickers = ['SPY', 'AAPL', 'MSFT']
data = yf.download(tickers, start='2019-01-01', end='2024-12-31')['Adj Close']

# Inspect the DatetimeIndex
print(data.index)
print(type(data.index))  # Should be DatetimeIndex
```

**What to check:** Is the index timezone-aware or naive? (Hint: check `data.index.tz`)

---

### **Task 2: Partial String Indexing Practice**

```python

# 1. Select all of Q3 2023 (July-September)
q3_2023 = data['2023-07-01':'2023-09-30']

# 2. Select all Fridays in 2023
# Friday = 4 (Monday=0, Tuesday=1, ..., Friday=4, Saturday=5, Sunday=6)
fridays_2023 = data[(data.index.year == 2023) & (data.index.dayofweek == 4)]

# 3. Last trading day per month in 2023
last_days = data['2023'].resample('M').last()

# Validation checks
print(f"Q3 2023 rows: {len(q3_2023)}")  # Should be ~63
print(f"Fridays in 2023: {len(fridays_2023)}")  # Should be ~52  
print(f"Last trading days per month: {len(last_days)}")  # Should be 12

# Inspect results
print("\nFirst few Q3 dates:")
print(q3_2023.head())

print("\nFirst few Fridays:")
print(fridays_2023.head())

print("\nLast trading days:")
print(last_days)
```

### **What to Check**

**Expected row counts:**
- **Q3 2023:** ~63 trading days (3 months × ~21 trading days/month)
- **Fridays in 2023:** ~52 (roughly one per week, excluding holidays)
- **Last trading days:** Exactly 12 (one per month)

**Key concepts applied:**
- **Partial string indexing:** `data['2023-07-01':'2023-09-30']` slices by date range
- **Boolean indexing:** `data.index.year == 2023` filters by year
- **Datetime accessor:** `.index.dayofweek` extracts day of week (0=Monday, 4=Friday)
- **Resampling preview:** `.resample('M').last()` groups by month-end, takes last value

---

#### **Task 3: Timezone Conversion**

```python
# yfinance data is timezone-naive — localize to UTC
data.index = data.index.tz_localize('UTC')

# Convert to US/Eastern (NYSE timezone)
data.index = data.index.tz_convert('US/Eastern')

# Check the result
print(data.index[0])  # Should show timezone info
```

**What changes:** Times shift by UTC offset. If original was 16:00 UTC, it becomes 11:00 US/Eastern (during DST).

---

#### **Task 4: Business Day Calendar Validation**

```python
from pandas.tseries.offsets import CustomBusinessDay
from pandas.tseries.holiday import USFederalHolidayCalendar

# Generate NYSE trading days for 2023
nyse_cal = CustomBusinessDay(calendar=USFederalHolidayCalendar())
trading_days_2023 = pd.date_range('2023-01-01', '2023-12-31', freq=nyse_cal)

# Check: are there any weekends or holidays in our data?
actual_days = data['2023'].index.date  # Convert to date (no time component)
expected_days = trading_days_2023.date

# Find mismatches
missing = set(expected_days) - set(actual_days)
extra = set(actual_days) - set(expected_days)

print(f"Missing days: {missing}")
print(f"Extra days (shouldn't exist): {extra}")
```

**Expected result:** `missing` might show early January (market closed), `extra` should be empty.

---

### **🔍 Self-Check Questions (With Hints)**

1. **What's the difference between timezone-naive and timezone-aware timestamps?**  
   *Hint: What happens when you try to subtract a UTC timestamp from a US/Eastern timestamp?*

2. **How do you select "all Mondays in 2023" efficiently?**  
   *Hint: Use `.index.dayofweek` — what integer represents Monday?*

3. **Why does `df.loc['2023-01-15']` work but `df.iloc['2023-01-15']` fail?**  
   *Hint: What does "label-based" vs "position-based" mean?*

4. **How do you handle daylight saving time changes?**  
   *Hint: Does `tz_convert()` handle this automatically, or do you need manual offsets?*

5. **What's the danger of assuming 252 trading days per year?**  
   *Hint: Think about holidays, market closures, leap years.*

---

### **📦 End-of-Unit Deliverable**

**Function:** `get_last_trading_day(year, month)`

**Spec:**
- **Input:** Year (int), Month (int, 1-12)
- **Output:** Last trading day of that month (as `pd.Timestamp`)
- **Requirements:**
  - Use NYSE calendar (exclude US federal holidays)
  - Handle edge cases: December 31st on weekend, month ends on holiday
  - Validate: Test against known NYSE closures (e.g., Thanksgiving 2023 was Nov 23, market closed)

**Starter code:**

```python
from pandas.tseries.offsets import CustomBusinessDay
from pandas.tseries.holiday import USFederalHolidayCalendar
import pandas as pd

def get_last_trading_day(year, month):
    """
    Returns the last trading day of the specified month.
    
    Args:
        year (int): Year
        month (int): Month (1-12)
    
    Returns:
        pd.Timestamp: Last trading day
    """
    # TODO: Implement
    pass

# Test cases
print(get_last_trading_day(2023, 11))  # Should be Nov 30 (Thanksgiving week)
print(get_last_trading_day(2023, 12))  # Should be Dec 29 (Christmas week)
```

**Success criteria:**
- Function returns correct date for at least 3 test cases
- Handles December 2023 correctly (Christmas on Monday, market closed 25th)
- No weekends or holidays in output

---

### **⚠️ Why You Can't Skip This**

Let me be blunt about what breaks if you half-ass this unit:

1. **Corporate actions:** Dividends are paid on ex-dividend dates. Off-by-one-day = wrong adjusted returns.
2. **Rolling windows:** `.rolling(21)` assumes 21 *periods*, not 21 *days*. If your index has gaps, your window is wrong.
3. **Resampling:** Silently fills forward over weekends. If you don't validate, you're backtesting on phantom data.
4. **Global portfolios:** London closes 5 hours before New York. Timezone errors create artificial arbitrage opportunities that don't exist in reality.

**The test:** Can you explain to a portfolio manager why their Sharpe ratio changed when you switched from daily to business-day resampling? If not, you're not ready for Module 4.